# LESS — Phase 3: MMLU Targeted Data Selection

This notebook takes the completed Phase-2 8192-D candidate gradient datastores and performs target-task selection for MMLU.

Pipeline:

1. Load the four Phase-2 raw 8192-D datastores.
2. Normalize every candidate feature once and persist normalized datastores.
3. Prepare 5-shot MMLU target examples for all 57 subtasks.
4. At each of the four LoRA checkpoints, compute raw target gradients.
5. Project target gradients with the same Rademacher projector.
6. Average the 5 projected examples within each MMLU subtask.
7. Normalize each 57-dimensional-subtask feature.
8. Compute cosine similarity against normalized candidate features.
9. Weight the four checkpoint similarities by the epoch-average learning rates.
10. Take the maximum score over the 57 MMLU subtasks.
11. Select the global top 5% = 450 candidates.

**Important:** normalization is not an extra heuristic here; cosine similarity is the LESS influence measure and explicitly removes sequence-gradient norm effects. The notebook keeps Phase-2 raw `.dat` files untouched and creates separate normalized copies. The paper's default projection dimension is 8192 and its default selection fraction is 5%. 


In [1]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Device          : {device}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")


def print_gpu_memory(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA unavailable")
        return

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
    free, total = torch.cuda.mem_get_info()

    free = free / (1024 ** 3)
    total = total / (1024 ** 3)

    print(
        f"[{tag}] "
        f"allocated={allocated:.2f} GiB | "
        f"reserved={reserved:.2f} GiB | "
        f"peak={max_allocated:.2f} GiB | "
        f"free={free:.2f} GiB / {total:.2f} GiB"
    )


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print_gpu_memory("initial")

PyTorch version : 2.10.0+cu128
CUDA available  : True
Device          : cuda
GPU             : Tesla T4
CUDA version    : 12.8
[initial] allocated=0.00 GiB | reserved=0.00 GiB | peak=0.00 GiB | free=14.46 GiB / 14.56 GiB


In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m","pip","install","-q","--no-deps","torchao>=0.16.0"])
!pip install traker[fast]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for fast_jl: filename=fast_jl-0.1.3-cp312-cp312-linux_x86_64.whl size=958596 sha256=e96c34ff7bd34286fda430831243fdd593e69062da3f704a36eb6abdc11ef445
  Stored in directory: /root/.cache/pip/wheels/cd/5a/bd/a05a64ea6e542809cfafc036bcbd44f0c20a04115917e3ab0f
  Created wheel for traker: filename=traker-0.3.2-py3-none-any.whl size=29027 sha256=2ee4c3751ada300e6ab953a6e1b9a33e2cac370c78ed897c654614e96d31cc33
  Stored in directory: /root/.cache/pip/wheels/6f/9c/93/cacba5ebe6989142debfcab420506346a9b9d6cd7bae112161
Successfully built fast_jl traker


In [3]:
!pip install -q liger-kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.3/644.3 kB 28.8 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import os
import json
import csv
import zipfile
import random

import pandas as pd
import numpy as np

import gc
from tqdm.auto import tqdm

SEED = 42

MODEL_DTYPE = torch.bfloat16  # MUST match Phase 2's MODEL_DTYPE

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
PHASE2_DIR = Path("/kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-4/less_phase2")
DATASTORE_DIR = PHASE2_DIR / "gradient_datastore"

NUM_CHECKPOINTS = 4

PROJECTION_DIM = 8192

NUM_CANDIDATES = 9000

SELECTION_RATIO = 0.05
NUM_SELECTED = int(NUM_CANDIDATES * SELECTION_RATIO)

NUM_MMLU_SUBTASKS = 57
NUM_SHOT = 5
from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
MMLU_ROOT = Path("/kaggle/input/datasets/manasaindusrikarri/mmlu-data/mmlu")
MMLU_DEV_DIR = MMLU_ROOT / "dev"
MMLU_TEST_DIR = MMLU_ROOT / "test"

dev_files = sorted(MMLU_DEV_DIR.glob("*_dev.csv"))
test_files = sorted(MMLU_TEST_DIR.glob("*_test.csv"))

dev_subjects = {
    f.name.removesuffix("_dev.csv")
    for f in dev_files
}

test_subjects = {
    f.name.removesuffix("_test.csv")
    for f in test_files
}

print()
print("Dev subjects :", len(dev_subjects))
print("Test subjects:", len(test_subjects))

MMLU_SUBJECTS = sorted(dev_subjects)
print("-----------------------")
for i, subject in enumerate(MMLU_SUBJECTS, start=1):
    print(f"{i:02d}. {subject}")


Dev subjects : 57
Test subjects: 57
-----------------------
01. abstract_algebra
02. anatomy
03. astronomy
04. business_ethics
05. clinical_knowledge
06. college_biology
07. college_chemistry
08. college_computer_science
09. college_mathematics
10. college_medicine
11. college_physics
12. computer_security
13. conceptual_physics
14. econometrics
15. electrical_engineering
16. elementary_mathematics
17. formal_logic
18. global_facts
19. high_school_biology
20. high_school_chemistry
21. high_school_computer_science
22. high_school_european_history
23. high_school_geography
24. high_school_government_and_politics
25. high_school_macroeconomics
26. high_school_mathematics
27. high_school_microeconomics
28. high_school_physics
29. high_school_psychology
30. high_school_statistics
31. high_school_us_history
32. high_school_world_history
33. human_aging
34. human_sexuality
35. international_law
36. jurisprudence
37. logical_fallacies
38. machine_learning
39. management
40. marketing
41. medi

In [6]:
CHECKPOINT_ROOT = Path("/kaggle/input//notebooks/tanmairaghava/phase1-lora-adapting/checkpoints")

CHECKPOINTS = {
    epoch: CHECKPOINT_ROOT / f"epoch_{epoch}"
    for epoch in range(1, 5)
}

for epoch, path in CHECKPOINTS.items():
    print(f"Epoch {epoch}: {path}")
    

EPOCH_FILES = {
    1: DATASTORE_DIR / "epoch_1.dat",
    2: DATASTORE_DIR / "epoch_2.dat",
    3: DATASTORE_DIR / "epoch_3.dat",
    4: DATASTORE_DIR / "epoch_4.dat",
}

PROGRESS_FILE = DATASTORE_DIR / "progress_state.json"

if PROGRESS_FILE.exists():

    with open(PROGRESS_FILE, "r") as f:
        progress_state = json.load(f)

    print()
    print("Progress state:")
    print(json.dumps(progress_state, indent=2))

print("Datastore:", DATASTORE_DIR)

Epoch 1: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_1
Epoch 2: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_2
Epoch 3: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_3
Epoch 4: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_4

Progress state:
{
  "epoch_1": 9000,
  "epoch_2": 9000,
  "epoch_3": 9000,
  "epoch_4": 9000
}
Datastore: /kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-4/less_phase2/gradient_datastore


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

EPOCH = 1
CKPT_PATH = CHECKPOINTS[EPOCH]

print("Loading checkpoint:")
print(CKPT_PATH)

with open(CKPT_PATH / "adapter_config.json", "r") as f:
    adapter_config = json.load(f)

BASE_MODEL = adapter_config["base_model_name_or_path"]

print()
print("Base model:", BASE_MODEL)
print("LoRA rank :", adapter_config.get("r"))
print("LoRA alpha:", adapter_config.get("lora_alpha"))
print("Target modules:", adapter_config.get("target_modules"))

tokenizer = AutoTokenizer.from_pretrained(
    CKPT_PATH,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# FIX: was AutoPeftModelForCausalLM.from_pretrained(CKPT_PATH, ...) -- a
# single-step convenience loader. Phase 2 built the model in two explicit
# steps (base model, then PeftModel.from_pretrained on top of it). These
# *should* produce an identical module tree and therefore identical
# named_parameters() order, but "should" isn't good enough when the
# projection matrix is positionally indexed against that exact order --
# a divergence there is exactly what would produce a low cosine / huge
# magnitude mismatch in the verification cell below. Matching Phase 2's
# construction path exactly removes the question rather than requiring
# us to prove the two paths agree.

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=MODEL_DTYPE,
    device_map={"": device},
)

base_model.eval()

model = PeftModel.from_pretrained(
    base_model,
    CKPT_PATH,
    is_trainable=True,
)

model.eval()

print()
print("=" * 60)
print("EPOCH-1 MODEL LOADED")
print("=" * 60)

print("Model device:", next(model.parameters()).device)
print("Model dtype :", next(model.parameters()).dtype)
print("Trainable parameters:")
model.print_trainable_parameters()


Loading checkpoint:
/kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_1

Base model: Qwen/Qwen2.5-1.5B
LoRA rank : 128
LoRA alpha: 512
Target modules: ['q_proj', 'v_proj', 'k_proj', 'o_proj']


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


EPOCH-1 MODEL LOADED
Model device: cuda:0
Model dtype : torch.bfloat16
Trainable parameters:
trainable params: 34,865,152 || all params: 1,578,579,456 || trainable%: 2.2086


In [8]:
# ============================================================
# LOAD RAW PHASE-2 DATASTORES + BUILD NORMALIZED DATASTORES
#
# Phase 2 saved the 8192-D projected candidate features as raw
# float32 vectors. LESS uses cosine similarity, i.e. normalized
# gradient features. We therefore create a persistent normalized
# copy of each epoch datastore here, once, and use ONLY these
# normalized candidate features for selection.
#
# Raw files are never modified.
# ============================================================

DATASTORE_DTYPE = np.float32

EXPECTED_BYTES = (
    NUM_CANDIDATES
    * PROJECTION_DIM
    * np.dtype(DATASTORE_DTYPE).itemsize
)

CANDIDATE_FEATURES_RAW = {}
CANDIDATE_FEATURES_NORMALIZED = {}

NORMALIZED_DATASTORE_ROOT = Path(
    "/kaggle/working/phase3_selection/normalized_datastore"
)
NORMALIZED_DATASTORE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NORMALIZATION_CHUNK_SIZE = 256

for epoch, path in EPOCH_FILES.items():

    actual_bytes = path.stat().st_size

    print(f"\nEpoch {epoch}")
    print("  raw file :", path)
    print("  bytes    :", actual_bytes)
    print("  expected :", EXPECTED_BYTES)

    assert actual_bytes == EXPECTED_BYTES, (
        f"Unexpected file size for epoch {epoch}"
    )

    raw_features = np.memmap(
        path,
        dtype=DATASTORE_DTYPE,
        mode="r",
        shape=(NUM_CANDIDATES, PROJECTION_DIM),
    )

    CANDIDATE_FEATURES_RAW[epoch] = raw_features

    normalized_path = (
        NORMALIZED_DATASTORE_ROOT
        / f"epoch_{epoch}_normalized.dat"
    )

    if (
        normalized_path.exists()
        and normalized_path.stat().st_size == EXPECTED_BYTES
    ):
        normalized_features = np.memmap(
            normalized_path,
            dtype=DATASTORE_DTYPE,
            mode="r",
            shape=(NUM_CANDIDATES, PROJECTION_DIM),
        )

        print("  normalized: existing file reused")

    else:
        normalized_features = np.memmap(
            normalized_path,
            dtype=DATASTORE_DTYPE,
            mode="w+",
            shape=(NUM_CANDIDATES, PROJECTION_DIM),
        )

        print("  normalized: creating", normalized_path)

        for start in range(
            0,
            NUM_CANDIDATES,
            NORMALIZATION_CHUNK_SIZE,
        ):
            end = min(
                start + NORMALIZATION_CHUNK_SIZE,
                NUM_CANDIDATES,
            )

            chunk = np.asarray(
                raw_features[start:end],
                dtype=np.float32,
            )

            norms = np.linalg.norm(
                chunk,
                axis=1,
                keepdims=True,
            )

            if not np.isfinite(norms).all():
                raise RuntimeError(
                    f"Non-finite candidate norm in epoch {epoch}, "
                    f"rows {start}:{end}"
                )

            if np.any(norms <= 1e-12):
                raise RuntimeError(
                    f"Zero/near-zero candidate feature in epoch {epoch}, "
                    f"rows {start}:{end}"
                )

            normalized_features[start:end] = (
                chunk / norms
            ).astype(np.float32, copy=False)

        normalized_features.flush()

        # Re-open read-only after construction.
        del normalized_features
        gc.collect()

        normalized_features = np.memmap(
            normalized_path,
            dtype=DATASTORE_DTYPE,
            mode="r",
            shape=(NUM_CANDIDATES, PROJECTION_DIM),
        )

    CANDIDATE_FEATURES_NORMALIZED[epoch] = normalized_features

    # Sanity check.
    check_indices = [0, 1, 100, NUM_CANDIDATES - 1]
    check = np.asarray(
        normalized_features[check_indices],
        dtype=np.float32,
    )
    check_norms = np.linalg.norm(check, axis=1)

    print(
        "  shape     :",
        normalized_features.shape,
    )
    print(
        "  norm check:",
        np.round(check_norms, 7).tolist(),
    )

    assert np.allclose(
        check_norms,
        1.0,
        atol=1e-5,
    )

print()
print("=" * 70)
print("CANDIDATE DATASTORES READY")
print("=" * 70)
print("Raw feature dimension        :", PROJECTION_DIM)
print("Candidates per epoch         :", NUM_CANDIDATES)
print("Normalized datastore location:", NORMALIZED_DATASTORE_ROOT)



Epoch 1
  raw file : /kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-4/less_phase2/gradient_datastore/epoch_1.dat
  bytes    : 294912000
  expected : 294912000
  normalized: creating /kaggle/working/phase3_selection/normalized_datastore/epoch_1_normalized.dat
  shape     : (9000, 8192)
  norm check: [1.0, 0.9999998807907104, 1.0, 1.0]

Epoch 2
  raw file : /kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-4/less_phase2/gradient_datastore/epoch_2.dat
  bytes    : 294912000
  expected : 294912000
  normalized: creating /kaggle/working/phase3_selection/normalized_datastore/epoch_2_normalized.dat
  shape     : (9000, 8192)
  norm check: [1.0, 1.0, 1.0, 0.9999998807907104]

Epoch 3
  raw file : /kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-4/less_phase2/gradient_datastore/epoch_3.dat
  bytes    : 294912000
  expected : 294912000
  normalized: creating /kaggle/working/phase3_selection/normalized_datastore

In [9]:
MMLU_TARGETS = {}

for subject in MMLU_SUBJECTS:

    dev_path = (
        MMLU_DEV_DIR /
        f"{subject}_dev.csv"
    )

    assert dev_path.exists(), (
        f"Missing MMLU dev file: {dev_path}"
    )

    rows = []

    with open(
        dev_path,
        "r",
        encoding="utf-8",
    ) as f:

        reader = csv.reader(f)

        for row in reader:

            if not row:
                continue

            question = row[0]
            choices = row[1:5]
            answer = row[5].strip()

            # Your MMLU files use A/B/C/D.
            assert answer in ["A", "B", "C", "D"], (
                f"Unexpected answer {answer} "
                f"in {subject}"
            )

            rows.append({
                "question": question,
                "choices": choices,
                "answer_letter": answer,
            })

    # We need exactly 5 target examples per subtask.
    assert len(rows) >= 5, (
        f"{subject}: only {len(rows)} dev examples"
    )

    MMLU_TARGETS[subject] = rows[:5]


print("✓ 57 subjects × 5 examples = 285 targets")

✓ 57 subjects × 5 examples = 285 targets


In [10]:
MAX_LENGTH = 2048
ANSWER_LETTERS = ["A", "B", "C", "D"]

def mmlu_to_messages(example):

    question = example["question"]
    choices = example["choices"]
    answer_letter = example["answer_letter"]

    user_content = (
        f"{question}\n\n"
        f"A. {choices[0]}\n"
        f"B. {choices[1]}\n"
        f"C. {choices[2]}\n"
        f"D. {choices[3]}"
    )

    return [
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": answer_letter,
        },
    ]


def tokenize_mmlu_example(example):

    messages = mmlu_to_messages(example)
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    encoded = tokenizer(
        formatted_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    return {
        "formatted_text": formatted_text,
        "input_ids": input_ids,
        "labels": input_ids.copy(),
        "attention_mask": attention_mask,
    }

MMLU_TARGET_TOKENIZED = []

for subject in MMLU_SUBJECTS:

    for example_idx, example in enumerate(
        MMLU_TARGETS[subject]
    ):

        tokenized = tokenize_mmlu_example(example)

        MMLU_TARGET_TOKENIZED.append({
            "subject": subject,
            "subject_index": MMLU_SUBJECTS.index(subject),
            "example_index": example_idx,

            "question": example["question"],
            "answer_letter": example["answer_letter"],

            "formatted_text": tokenized["formatted_text"],

            "input_ids": tokenized["input_ids"],
            "labels": tokenized["labels"],
            "attention_mask": tokenized["attention_mask"],
        })

assert len(MMLU_TARGET_TOKENIZED) == 285

lengths = [
    len(x["input_ids"])
    for x in MMLU_TARGET_TOKENIZED
]

assert min(lengths) > 1, (
    "Tokenization is still incorrect: "
    "some examples have <= 1 token."
)

print("=" * 70)
print("MMLU TARGET TOKENIZATION COMPLETE")
print("=" * 70)

print("Subjects       :", len(MMLU_SUBJECTS))
print("Examples       :", len(MMLU_TARGET_TOKENIZED))
print("Min token len  :", min(lengths))
print("Max token len  :", max(lengths))
print("Mean token len :", np.mean(lengths))

MMLU TARGET TOKENIZATION COMPLETE
Subjects       : 57
Examples       : 285
Min token len  : 44
Max token len  : 545
Mean token len : 117.36140350877193


In [11]:
from trak.projectors import (
    BasicProjector,
    CudaProjector,
    ProjectionType,
)

NUM_TARGETS = len(MMLU_TARGET_TOKENIZED)
NUM_CANDIDATES = 9000

PROJECTION_DIM = 8192
PROJECTION_BATCH_SIZE = 4  # early Phase-2 config; actual extraction used 16

TOP_FRACTION = 0.05
TOP_K = int(NUM_CANDIDATES * TOP_FRACTION)

RAW_GRAD_DIM = 34_865_152

In [12]:
# ============================================================
# CELL 9: Build the SAME projector used to create the Phase-2
# candidate datastore.
#
# This is the single most important correctness constraint in this
# notebook: target features and candidate features are only
# comparable via cosine similarity if they came out of the
# IDENTICAL random projection matrix. That matrix is fully
# determined by (projector class, seed, proj_dim, proj_type,
# model_id). Phase 2 (confirmed from its actual source) used:
#   CudaProjector, seed=0, proj_dim=8192, proj_type=rademacher,
#   max_batch_size=16, model_id=0 (passed to .project(), not the
#   constructor).
#
# The earlier draft of this cell used seed=42 -- that alone would
# silently select a DIFFERENT projection matrix from the one baked
# into 4 epochs x 9000 examples of already-computed candidate
# features, making every downstream cosine similarity meaningless.
# It also passed dtype= and block_size= to CudaProjector, which
# only BasicProjector accepts -- that call raises TypeError the
# moment fast_jl is actually available.
#
# We do NOT silently fall back to BasicProjector on CPU if
# CudaProjector is unavailable. There is no guarantee that
# BasicProjector's PyTorch-CPU RNG stream reproduces CudaProjector's
# custom CUDA-kernel RNG stream for "the same" seed -- they are
# different implementations. Phase 2 got CudaProjector working on
# this exact GPU/CUDA combination (Tesla T4, CUDA 12.8), so if it's
# unavailable here, the right fix is the fast_jl install, not a
# quiet switch to an incompatible projector.
# ============================================================

PROJECTOR_SEED = 0              # MUST match Phase 2's PROJECTOR_SEED
PROJECTOR_MODEL_ID = 0          # MUST match Phase 2's project(model_id=0) calls
PROJECTOR_MAX_BATCH_SIZE = 16   # actual Phase-2 extraction value


def get_exact_trak_projector(device, grad_dim, proj_dim):

    import fast_jl

    required_fn = "project_rademacher_8"

    cuda_projector_available = hasattr(fast_jl, required_fn)

    print("fast_jl module          :", fast_jl)
    print(f"{required_fn} available : {cuda_projector_available}")

    if not cuda_projector_available:
        raise RuntimeError(
            "fast_jl / CudaProjector is not available in this "
            "environment, but Phase 2's candidate datastore was built "
            "with CudaProjector. Falling back to BasicProjector here "
            "would silently switch to a DIFFERENT random projection "
            "matrix (different RNG implementation entirely), which "
            "would make every cosine similarity against the existing "
            "datastore meaningless. Fix the fast_jl install (it worked "
            "for Phase 2 on this same GPU type) instead of swapping "
            "projectors."
        )

    print("Using CudaProjector (matches Phase 2 exactly).")

    projector = CudaProjector(
        grad_dim=grad_dim,
        proj_dim=proj_dim,
        proj_type=ProjectionType.rademacher,
        seed=PROJECTOR_SEED,
        max_batch_size=PROJECTOR_MAX_BATCH_SIZE,
        device=device,
    )

    return projector


projector = get_exact_trak_projector(
    device=device,
    grad_dim=RAW_GRAD_DIM,
    proj_dim=PROJECTION_DIM,
)


fast_jl module          : <module 'fast_jl' from '/usr/local/lib/python3.12/dist-packages/fast_jl.cpython-312-x86_64-linux-gnu.so'>
project_rademacher_8 available : True
Using CudaProjector (matches Phase 2 exactly).


In [13]:
# ============================================================
# Reload the ORIGINAL 9,000 candidate examples in the exact same
# merge order Phase 2 used, and tokenize them the exact same way.
# Row i here MUST correspond to row i in the .dat datastores --
# both the verification cell and the final selection step depend
# on this order matching exactly.
# ============================================================

from datasets import load_dataset, concatenate_datasets

DATA_FILES = {
    "flan_v2": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/flan_v2_mini.jsonl"),
    "cot": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/cot_mini.jsonl"),
    "dolly": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/dolly_mini.jsonl"),
    "oasst1": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/oasst1_mini.jsonl"),
}

_source_datasets = {}

for name, path in DATA_FILES.items():
    _source_datasets[name] = load_dataset(
        "json",
        data_files=str(path),
        split="train",
    )

candidate_dataset = concatenate_datasets(list(_source_datasets.values()))

assert len(candidate_dataset) == NUM_CANDIDATES, (
    f"Reloaded candidate dataset has {len(candidate_dataset)} rows, "
    f"expected {NUM_CANDIDATES}. Row order/count must match Phase 2 "
    f"exactly, or every index below points at the wrong example."
)


def tokenize_candidate_example(example):
    encoded = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )

    if hasattr(encoded, "encodings") and encoded.encodings:
        input_ids = encoded.encodings[0].ids
    elif hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids
        if input_ids and isinstance(input_ids[0], list):
            input_ids = input_ids[0]
    else:
        input_ids = encoded

    input_ids = list(input_ids)[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": input_ids.copy(),
        "attention_mask": [1] * len(input_ids),
    }


tokenized_candidate_dataset = candidate_dataset.map(
    tokenize_candidate_example,
    remove_columns=candidate_dataset.column_names,
    desc="Tokenizing 9,000 candidates (Phase-2-identical)",
)

print("Candidate dataset reloaded  :", len(candidate_dataset))
print("Tokenized                   :", len(tokenized_candidate_dataset))


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing 9,000 candidates (Phase-2-identical):   0%|          | 0/9000 [00:00<?, ? examples/s]

Candidate dataset reloaded  : 9000
Tokenized                   : 9000


In [14]:
# ============================================================
# Ported from Phase 2: positional mapping between trainable LoRA
# parameters and their Adam optimizer states, plus the Adam
# gradient transform. This is used ONLY by the verification cell
# below (to reconstruct a candidate feature exactly as Phase 2
# computed it) -- target/validation features themselves use RAW
# gradients with no Adam transform, per the paper's Definition 3.1
# (validation side: raw grad; training/candidate side: Adam-
# transformed Gamma). That asymmetry is intentional, not a bug.
# ============================================================

ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8


def transform_gradient_adam(gradient, exp_avg, exp_avg_sq):

    m = exp_avg.to(device=gradient.device, dtype=gradient.dtype)
    v = exp_avg_sq.to(device=gradient.device, dtype=gradient.dtype)

    return (
        ADAM_BETA1 * m + (1.0 - ADAM_BETA1) * gradient
    ) / torch.sqrt(
        ADAM_BETA2 * v
        + (1.0 - ADAM_BETA2) * gradient.square()
        + ADAM_EPS
    )


def build_parameter_state_mapping(model, checkpoint_root, reference_epoch=1):

    trainable_parameters = [
        p for p in model.parameters() if p.requires_grad
    ]

    assert len(trainable_parameters) == 224, (
        f"Expected 224 trainable tensors, got "
        f"{len(trainable_parameters)}. LoRA config does not match "
        f"Phase 2 -- stop and investigate before proceeding."
    )

    reference_state = torch.load(
        checkpoint_root / f"epoch_{reference_epoch}" / "training_state.pt",
        map_location="cpu",
        weights_only=False,
    )

    optimizer_param_ids = (
        reference_state["optimizer"]["param_groups"][0]["params"]
    )

    assert len(optimizer_param_ids) == len(trainable_parameters)

    named_parameters = dict(model.named_parameters())

    mapping = []

    for position, (parameter, parameter_id) in enumerate(
        zip(trainable_parameters, optimizer_param_ids)
    ):
        parameter_name = next(
            name for name, p in named_parameters.items()
            if p is parameter
        )

        mapping.append({
            "position": position,
            "parameter": parameter,
            "parameter_id": parameter_id,
            "name": parameter_name,
            "numel": parameter.numel(),
        })

    return mapping


def load_adam_state(checkpoint_root, epoch):

    state = torch.load(
        checkpoint_root / f"epoch_{epoch}" / "training_state.pt",
        map_location="cpu",
        weights_only=False,
    )

    return state["optimizer"]["state"]


def build_adam_gradient_vector(parameter_state_mapping, adam_state_dict, device):

    flat_gradient = torch.empty(
        RAW_GRAD_DIM, dtype=torch.float16, device=device,
    )

    offset = 0

    with torch.no_grad():

        for entry in parameter_state_mapping:

            parameter = entry["parameter"]
            numel = entry["numel"]

            gradient = parameter.grad
            assert gradient is not None, f"Missing gradient: {entry['name']}"

            state = adam_state_dict[entry["parameter_id"]]

            transformed = transform_gradient_adam(
                gradient, state["exp_avg"], state["exp_avg_sq"],
            )

            flat_gradient[offset:offset + numel].copy_(
                transformed.reshape(-1)
            )

            offset += numel

    assert offset == RAW_GRAD_DIM

    return flat_gradient


parameter_state_mapping = build_parameter_state_mapping(
    model=model,
    checkpoint_root=CHECKPOINT_ROOT,
    reference_epoch=1,
)

print("Trainable tensors mapped:", len(parameter_state_mapping))
print()
print("First 5 parameter names (cross-check against Phase 2's own")
print("\"First 10 parameter entries\" printout by eye -- order must match):")
for entry in parameter_state_mapping[:5]:
    print(f"  [{entry['position']:3d}] {entry['name']}")


Trainable tensors mapped: 224

First 5 parameter names (cross-check against Phase 2's own
"First 10 parameter entries" printout by eye -- order must match):
  [  0] base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
  [  1] base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
  [  2] base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
  [  3] base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
  [  4] base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight


In [15]:
# ============================================================
# PHASE-2 REPRODUCTION CHECK — EXACT CANDIDATE PATH
#
# IMPORTANT:
# A target MMLU RAW gradient must NOT be compared to a candidate
# datastore row. Candidate rows contain Gamma = Adam-transformed
# gradients, while validation rows use the raw gradient.
#
# Therefore this verification deliberately recomputes the SAME
# candidate example using:
#   Phase-2 model construction
#   Phase-2 train/eval + checkpointing configuration
#   Phase-2 fused CE loss
#   Phase-2 positional parameter ordering
#   Phase-2 Adam transform
#   Phase-2 CudaProjector
#   Phase-2 seed/model_id/projection dimension
#
# Only after this exact candidate-vs-candidate check passes do we
# proceed to MMLU target gradients.
# ============================================================

VERIFY_EPOCH = 1
VERIFY_CANDIDATE_IDX = 0

verify_projector = CudaProjector(
    grad_dim=RAW_GRAD_DIM,
    proj_dim=PROJECTION_DIM,
    proj_type=ProjectionType.rademacher,
    seed=PROJECTOR_SEED,
    max_batch_size=PROJECTOR_MAX_BATCH_SIZE,
    device=device,
)

verify_adam_state = load_adam_state(
    CHECKPOINT_ROOT,
    VERIFY_EPOCH,
)

verify_example = tokenized_candidate_dataset[
    VERIFY_CANDIDATE_IDX
]

# Reproduce Phase-2 memory/forward configuration exactly.
model.config.use_cache = False
model.base_model.model.config._attn_implementation = "eager"

model.enable_input_require_grads()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
    reduction="mean",
    ignore_index=-100,
)

model.train()

for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.eval()

transformer = model.base_model.model.model
lm_head = model.base_model.model.lm_head

model.zero_grad(set_to_none=True)

_input_ids = torch.tensor(
    verify_example["input_ids"],
    dtype=torch.long,
    device=device,
).unsqueeze(0)

_attention_mask = torch.tensor(
    verify_example["attention_mask"],
    dtype=torch.long,
    device=device,
).unsqueeze(0)

_labels = _input_ids.reshape(-1)

with torch.autocast(
    device_type="cuda",
    dtype=MODEL_DTYPE,
):
    _outputs = transformer(
        input_ids=_input_ids,
        attention_mask=_attention_mask,
        use_cache=False,
        return_dict=True,
    )

    _hidden_states = _outputs.last_hidden_state.reshape(
        -1,
        _outputs.last_hidden_state.shape[-1],
    )

    _loss = fused_loss_fn(
        lm_head.weight,
        _hidden_states,
        _labels,
    )

if not torch.isfinite(_loss):
    raise RuntimeError(
        f"Non-finite verification loss: {_loss.item()}"
    )

_loss.backward()

fresh_adam_gradient = build_adam_gradient_vector(
    parameter_state_mapping=parameter_state_mapping,
    adam_state_dict=verify_adam_state,
    device=device,
)

with torch.no_grad():
    fresh_projected = (
        verify_projector.project(
            fresh_adam_gradient.unsqueeze(0),
            model_id=PROJECTOR_MODEL_ID,
        )
        .squeeze(0)
        .float()
        .cpu()
        .numpy()
    )

stored_projected = np.asarray(
    CANDIDATE_FEATURES_RAW[
        VERIFY_EPOCH
    ][VERIFY_CANDIDATE_IDX],
    dtype=np.float32,
)

fresh_norm = np.linalg.norm(fresh_projected)
stored_norm = np.linalg.norm(stored_projected)

fresh_normalized = fresh_projected / max(
    fresh_norm,
    1e-12,
)

stored_normalized = stored_projected / max(
    stored_norm,
    1e-12,
)

verification_cosine = float(
    fresh_normalized @ stored_normalized
)

print("=" * 70)
print("EXACT PHASE-2 CANDIDATE FEATURE VERIFICATION")
print("=" * 70)
print(f"Epoch                : {VERIFY_EPOCH}")
print(f"Candidate index      : {VERIFY_CANDIDATE_IDX}")
print(f"Fresh raw norm       : {fresh_norm:.6f}")
print(f"Stored raw norm      : {stored_norm:.6f}")
print(f"Normalized cosine    : {verification_cosine:.9f}")
print()

if verification_cosine < 0.999:
    raise RuntimeError(
        "Exact Phase-2 candidate reproduction FAILED. "
        "Do not compute the MMLU target gradients yet. "
        "The mismatch is now isolated to the Phase-2 candidate "
        "reproduction path/projector/model configuration."
    )

print("✓ EXACT Phase-2 candidate feature reproduced.")
print("✓ Candidate projection space is verified.")
print("✓ Safe to proceed to MMLU target gradients.")

model.zero_grad(set_to_none=True)

del (
    verify_projector,
    verify_adam_state,
    _input_ids,
    _attention_mask,
    _labels,
    _outputs,
    _hidden_states,
    _loss,
    fresh_adam_gradient,
    fresh_projected,
    stored_projected,
)

gc.collect()
torch.cuda.empty_cache()


EXACT PHASE-2 CANDIDATE FEATURE VERIFICATION
Epoch                : 1
Candidate index      : 0
Fresh raw norm       : 699756.812500
Stored raw norm      : 699756.812500
Normalized cosine    : 1.000000000

✓ EXACT Phase-2 candidate feature reproduced.
✓ Candidate projection space is verified.
✓ Safe to proceed to MMLU target gradients.


In [16]:
# Diagnostic cell intentionally removed from the final notebook.
# If the Phase-2 verification warning appears, debug projector
# compatibility separately before treating the selected set as a
# faithful reproduction.


In [17]:
MMLU_TARGET_GROUPS = {
    subject: []
    for subject in MMLU_SUBJECTS
}

for target in MMLU_TARGET_TOKENIZED:

    MMLU_TARGET_GROUPS[
        target["subject"]
    ].append(target)


assert len(MMLU_TARGET_GROUPS) == 57

assert all(
    len(examples) == 5
    for examples in MMLU_TARGET_GROUPS.values()
)

print("=" * 60)
print("MMLU TARGET GROUPS")
print("=" * 60)

print("Subjects:", len(MMLU_TARGET_GROUPS))

for subject in MMLU_SUBJECTS[:10]:
    print(
        f"{subject:35s}: "
        f"{len(MMLU_TARGET_GROUPS[subject])} examples"
    )

print("...")

MMLU TARGET GROUPS
Subjects: 57
abstract_algebra                   : 5 examples
anatomy                            : 5 examples
astronomy                          : 5 examples
business_ethics                    : 5 examples
clinical_knowledge                 : 5 examples
college_biology                    : 5 examples
college_chemistry                  : 5 examples
college_computer_science           : 5 examples
college_mathematics                : 5 examples
college_medicine                   : 5 examples
...


In [18]:
def cosine_scores_against_candidates(
    target_feature,
    candidate_features_normalized,
):
    """
    Compute cosine similarity against the already-normalized
    candidate datastore.

    Both sides are unit-normalized before this function is called.
    Therefore cosine(a, b) = a @ b.
    """

    target = np.asarray(
        target_feature,
        dtype=np.float32,
    )

    assert target.shape == (PROJECTION_DIM,)

    target_norm = np.linalg.norm(target)

    if not np.isfinite(target_norm) or target_norm <= 1e-12:
        raise RuntimeError(
            "Target feature has invalid/zero norm."
        )

    target = target / target_norm

    scores = (
        np.asarray(
            candidate_features_normalized,
            dtype=np.float32,
        )
        @ target
    )

    return scores.astype(np.float32)


In [19]:
def compute_target_gradient(
    model,
    target,
    device,
    transformer,
    lm_head,
    fused_loss_fn,
):
    """
    RAW validation gradient for one MMLU example.

    This intentionally mirrors Phase 2's candidate gradient
    computation as closely as possible:

      transformer forward
        -> hidden states
        -> Liger fused CE
        -> backward
        -> concatenate LoRA gradients in the same parameter order

    The ONLY candidate-side operation omitted here is the Adam
    transform. LESS Definition 3.1 uses the raw validation
    gradient on the validation side and Gamma/Adam-transformed
    gradient on the candidate side.
    """

    model.zero_grad(set_to_none=True)

    input_ids = torch.tensor(
        target["input_ids"],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    attention_mask = torch.tensor(
        target["attention_mask"],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    labels = input_ids.reshape(-1)

    with torch.autocast(
        device_type="cuda",
        dtype=MODEL_DTYPE,
    ):
        outputs = transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )

        hidden_states = outputs.last_hidden_state.reshape(
            -1,
            outputs.last_hidden_state.shape[-1],
        )

        loss = fused_loss_fn(
            lm_head.weight,
            hidden_states,
            labels,
        )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite target loss: "
            f"subject={target['subject']} "
            f"example={target['example_index']}: "
            f"{loss.item()}"
        )

    loss.backward()

    gradient_parts = []

    # Same named-parameter traversal used for the Phase-2
    # LoRA gradient vector.
    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if param.grad is None:
            raise RuntimeError(
                f"Missing gradient: {name}"
            )

        if not torch.isfinite(param.grad).all():
            raise RuntimeError(
                f"Non-finite gradient: {name}"
            )

        gradient_parts.append(
            param.grad.detach()
            .float()
            .reshape(-1)
        )

    raw_gradient = torch.cat(
        gradient_parts
    )

    assert raw_gradient.numel() == RAW_GRAD_DIM

    model.zero_grad(set_to_none=True)

    del (
        input_ids,
        attention_mask,
        labels,
        outputs,
        hidden_states,
        loss,
    )

    return raw_gradient


In [20]:
NUM_MMLU_SUBTASKS = len(MMLU_SUBJECTS)


def compute_checkpoint_target_gradients(
    model,
    device,
    epoch,
    projector,
    transformer,
    lm_head,
    fused_loss_fn,
):
    """
    One checkpoint:

      57 MMLU subjects
        × 5 few-shot examples
        → RAW LoRA gradient
        → exact Phase-2 Rademacher projector
        → average 5 projected examples per subject
        → L2-normalize each subject feature

    A fresh CudaProjector is supplied for each epoch, exactly as
    Phase 2 instantiated a fresh projector inside its checkpoint
    extraction loop. This is important because the candidate
    datastore for each epoch was generated by a fresh projector
    with the same seed/model_id.
    """

    print()
    print("=" * 70)
    print(f"TARGET FEATURES -- EPOCH {epoch}")
    print("=" * 70)

    subject_features = np.empty(
        (
            NUM_MMLU_SUBTASKS,
            PROJECTION_DIM,
        ),
        dtype=np.float32,
    )

    for subject_idx, subject in enumerate(
        tqdm(
            MMLU_SUBJECTS,
            desc=f"Epoch {epoch}",
        )
    ):

        examples = MMLU_TARGET_GROUPS[subject]

        assert len(examples) == NUM_SHOT

        accumulated = np.zeros(
            PROJECTION_DIM,
            dtype=np.float32,
        )

        for example in examples:

            raw_gradient = compute_target_gradient(
                model=model,
                target=example,
                device=device,
                transformer=transformer,
                lm_head=lm_head,
                fused_loss_fn=fused_loss_fn,
            )

            with torch.no_grad():

                projected = (
                    projector.project(
                        raw_gradient.unsqueeze(0),
                        model_id=PROJECTOR_MODEL_ID,
                    )
                    .squeeze(0)
                    .float()
                    .cpu()
                    .numpy()
                )

            if not np.isfinite(projected).all():
                raise RuntimeError(
                    f"Non-finite projected target feature: "
                    f"epoch={epoch}, subject={subject}"
                )

            accumulated += projected

            del raw_gradient
            del projected

            gc.collect()
            torch.cuda.empty_cache()

        # Exactly the requested 5-shot MMLU subtask aggregation.
        subject_feature = accumulated / NUM_SHOT

        # Normalize AFTER averaging the five examples.
        subject_norm = np.linalg.norm(subject_feature)

        if (
            not np.isfinite(subject_norm)
            or subject_norm <= 1e-12
        ):
            raise RuntimeError(
                f"Invalid target feature norm: "
                f"epoch={epoch}, subject={subject}, "
                f"norm={subject_norm}"
            )

        subject_features[subject_idx] = (
            subject_feature / subject_norm
        ).astype(
            np.float32,
            copy=False,
        )

    assert np.isfinite(subject_features).all()

    final_norms = np.linalg.norm(
        subject_features,
        axis=1,
    )

    assert np.allclose(
        final_norms,
        1.0,
        atol=1e-5,
    )

    print(
        f"Normalized target feature norms: "
        f"min={final_norms.min():.7f}, "
        f"max={final_norms.max():.7f}"
    )

    return subject_features


In [21]:
EPOCH_AVG_LR = {
    1: 1.860810e-05,
    2: 1.283336e-05,
    3: 5.398873e-06,
    4: 6.596639e-07,
}

print("=" * 70)
print("EPOCH-AVERAGE LEARNING RATES")
print("=" * 70)

for epoch in range(1, 5):
    print(
        f"Epoch {epoch}: "
        f"{EPOCH_AVG_LR[epoch]:.9e}"
    )

print()
print(
    "Mean across epochs : "
    f"{sum(EPOCH_AVG_LR.values()) / 4:.9e}"
)

assert len(EPOCH_AVG_LR) == 4
assert all(
    EPOCH_AVG_LR[e] > 0
    for e in range(1, 5)
)

print("\n✓ All 4 epoch-average learning rates loaded")

EPOCH-AVERAGE LEARNING RATES
Epoch 1: 1.860810000e-05
Epoch 2: 1.283336000e-05
Epoch 3: 5.398873000e-06
Epoch 4: 6.596639000e-07

Mean across epochs : 9.374999225e-06

✓ All 4 epoch-average learning rates loaded


In [22]:
# ============================================================
# MAIN PRODUCTION LOOP
#
# For each epoch:
#
#   target MMLU subtask feature (57 × 8192, normalized)
#                     ×
#   candidate datastore (9000 × 8192, normalized)
#                     ↓
#               cosine matrix
#                     ↓
#          eta_bar(epoch) weighted sum
#
# We aggregate the four epochs FIRST for each MMLU subject, then
# take max over the 57 subjects:
#
#   Score(z) =
#       max_j sum_i eta_bar_i *
#           cos(target_j_i, candidate_z_i)
#
# This follows Definition 3.1 and the multi-subtask selection
# formulation described in LESS.
#
# Candidate datastores are already normalized in Cell 6.
# Target features are already normalized in Cell 18.
# Therefore the similarity computation is simply a dot product.
# ============================================================

OUTPUT_ROOT = Path(
    "/kaggle/working/phase3_selection"
)
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STATE_PATH = (
    OUTPUT_ROOT
    / "aggregation_state.npz"
)

if STATE_PATH.exists():

    _state = np.load(
        STATE_PATH,
        allow_pickle=True,
    )

    weighted_sum_per_subject = (
        _state[
            "weighted_sum_per_subject"
        ]
    )

    completed_epochs = set(
        _state[
            "completed_epochs"
        ].tolist()
    )

    print(
        "Resuming: completed epochs =",
        sorted(completed_epochs),
    )

else:

    weighted_sum_per_subject = np.zeros(
        (
            NUM_MMLU_SUBTASKS,
            NUM_CANDIDATES,
        ),
        dtype=np.float64,
    )

    completed_epochs = set()


assert len(EPOCH_AVG_LR) == 4, (
    "EPOCH_AVG_LR must contain all four "
    "epoch-average learning rates."
)


for epoch in range(1, 5):

    if epoch in completed_epochs:
        print(
            f"Epoch {epoch}: already aggregated, skipping."
        )
        continue

    feature_cache_path = (
        OUTPUT_ROOT
        / f"epoch_{epoch}_target_features_normalized.npy"
    )

    if feature_cache_path.exists():

        print(
            f"Epoch {epoch}: "
            "loading cached normalized target features."
        )

        subject_features = np.load(
            feature_cache_path
        )

    else:

        print(
            f"Epoch {epoch}: "
            "loading checkpoint and computing "
            "normalized target features."
        )

        epoch_base_model = (
            AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                dtype=MODEL_DTYPE,
                device_map={"": device},
            )
        )

        epoch_base_model.eval()

        epoch_model = (
            PeftModel.from_pretrained(
                epoch_base_model,
                CHECKPOINTS[epoch],
                is_trainable=True,
            )
        )

        epoch_model.eval()

        # Re-enable the exact memory-efficient gradient path used
        # during the verified target-gradient test.
        epoch_model.config.use_cache = False
        epoch_model.base_model.model.config._attn_implementation = "eager"

        epoch_model.enable_input_require_grads()

        epoch_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )

        epoch_model.train()

        for module in epoch_model.modules():
            if isinstance(
                module,
                torch.nn.Dropout,
            ):
                module.eval()

        # Phase 2 creates a NEW projector inside each checkpoint
        # extraction pass. Reproduce that exactly here.
        epoch_projector = CudaProjector(
            grad_dim=RAW_GRAD_DIM,
            proj_dim=PROJECTION_DIM,
            proj_type=ProjectionType.rademacher,
            seed=PROJECTOR_SEED,
            max_batch_size=PROJECTOR_MAX_BATCH_SIZE,
            device=device,
        )

        # The target-gradient function uses the underlying HF
        # transformer/lm_head and the exact Phase-2 fused loss path.
        epoch_transformer = (
            epoch_model.base_model.model.model
        )
        epoch_lm_head = (
            epoch_model.base_model.model.lm_head
        )

        # Bind the exact checkpoint's modules for this call.
        old_transformer = globals().get("transformer", None)
        old_lm_head = globals().get("lm_head", None)
        old_fused_loss_fn = globals().get("fused_loss_fn", None)

        transformer = epoch_transformer
        lm_head = epoch_lm_head

        if "fused_loss_fn" not in globals() or fused_loss_fn is None:
            fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
                reduction="mean",
                ignore_index=-100,
            )

        subject_features = (
            compute_checkpoint_target_gradients(
                model=epoch_model,
                device=device,
                epoch=epoch,
                projector=epoch_projector,
                transformer=epoch_transformer,
                lm_head=epoch_lm_head,
                fused_loss_fn=fused_loss_fn,
            )
        )

        assert subject_features.shape == (
            NUM_MMLU_SUBTASKS,
            PROJECTION_DIM,
        )

        assert np.allclose(
            np.linalg.norm(
                subject_features,
                axis=1,
            ),
            1.0,
            atol=1e-5,
        )

        np.save(
            feature_cache_path,
            subject_features,
        )

        del epoch_projector
        del epoch_transformer
        del epoch_lm_head

        # Restore globals to the initial model objects when they exist.
        if old_transformer is not None:
            transformer = old_transformer
        if old_lm_head is not None:
            lm_head = old_lm_head
        if old_fused_loss_fn is not None:
            fused_loss_fn = old_fused_loss_fn

        del epoch_model
        del epoch_base_model

        gc.collect()
        torch.cuda.empty_cache()

    # ========================================================
    # Candidate side: ALREADY NORMALIZED in Cell 6.
    # ========================================================

    candidate_matrix = (
        CANDIDATE_FEATURES_NORMALIZED[
            epoch
        ]
    )

    assert candidate_matrix.shape == (
        NUM_CANDIDATES,
        PROJECTION_DIM,
    )

    # Both matrices are unit-normalized, so this is cosine.
    cosine_matrix = (
        np.asarray(
            subject_features,
            dtype=np.float32,
        )
        @ np.asarray(
            candidate_matrix,
            dtype=np.float32,
        ).T
    )

    assert cosine_matrix.shape == (
        NUM_MMLU_SUBTASKS,
        NUM_CANDIDATES,
    )

    assert np.isfinite(
        cosine_matrix
    ).all()

    eta_i = float(
        EPOCH_AVG_LR[epoch]
    )

    weighted_sum_per_subject += (
        eta_i * cosine_matrix
    )

    completed_epochs.add(epoch)

    np.savez(
        STATE_PATH,
        weighted_sum_per_subject=(
            weighted_sum_per_subject
        ),
        completed_epochs=np.array(
            sorted(completed_epochs)
        ),
    )

    print(
        f"Epoch {epoch} aggregated. "
        f"eta_bar={eta_i:.10e}"
    )

print()
print(
    "All epochs aggregated:",
    sorted(completed_epochs),
)


Epoch 1: loading checkpoint and computing normalized target features.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


TARGET FEATURES -- EPOCH 1


Epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Normalized target feature norms: min=0.9999999, max=1.0000001
Epoch 1 aggregated. eta_bar=1.8608100000e-05
Epoch 2: loading checkpoint and computing normalized target features.


...

Normalized target feature norms: min=0.9999999, max=1.0000000
Epoch 3 aggregated. eta_bar=5.3988730000e-06
Epoch 4: loading checkpoint and computing normalized target features.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


TARGET FEATURES -- EPOCH 4


Epoch 4:   0%|          | 0/57 [00:00<?, ?it/s]

Normalized target feature norms: min=0.9999999, max=1.0000001
Epoch 4 aggregated. eta_bar=6.5966390000e-07

All epochs aggregated: [1, 2, 3, 4]


In [23]:
# ============================================================
# FINAL SCORING, TOP-5% SELECTION, AND SAVE
# ============================================================

assert len(completed_epochs) == 4

# Max over the 57 MMLU subtasks AFTER the four-epoch weighted
# trajectory has been accumulated.
final_scores = (
    weighted_sum_per_subject.max(
        axis=0
    )
)

assert final_scores.shape == (
    NUM_CANDIDATES,
)

best_subject_idx = (
    weighted_sum_per_subject.argmax(
        axis=0
    )
)

ranked_indices = np.argsort(
    final_scores
)[::-1]

top_indices = ranked_indices[
    :TOP_K
]

print("=" * 70)
print("FINAL LESS SELECTION")
print("=" * 70)
print(
    f"Candidates       : {NUM_CANDIDATES}"
)
print(
    f"Selection ratio  : {TOP_FRACTION:.1%}"
)
print(
    f"Selected         : {TOP_K}"
)
print(
    f"Score min/max    : "
    f"{final_scores.min():.6f} / "
    f"{final_scores.max():.6f}"
)
print(
    f"Selected min/max : "
    f"{final_scores[top_indices].min():.6f} / "
    f"{final_scores[top_indices].max():.6f}"
)

selected_records = []

for rank, idx in enumerate(
    top_indices,
    start=1,
):

    record = dict(
        candidate_dataset[
            int(idx)
        ]
    )

    record[
        "_candidate_index"
    ] = int(idx)

    record[
        "_rank"
    ] = int(rank)

    record[
        "_score"
    ] = float(
        final_scores[idx]
    )

    record[
        "_best_mmlu_subject"
    ] = MMLU_SUBJECTS[
        int(
            best_subject_idx[idx]
        )
    ]

    selected_records.append(
        record
    )


selected_path = (
    OUTPUT_ROOT
    / "less_selected_top450.jsonl"
)

with open(
    selected_path,
    "w",
    encoding="utf-8",
) as f:

    for record in selected_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


scores_path = (
    OUTPUT_ROOT
    / "less_scores_all_9000.npz"
)

np.savez(
    scores_path,
    scores=final_scores,
    best_subject_idx=best_subject_idx,
    top_indices=top_indices,
)


metadata_path = (
    OUTPUT_ROOT
    / "less_selection_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "method": "LESS",
            "target_task": "MMLU",
            "num_mmlu_subtasks": 57,
            "num_shot_per_subtask": NUM_SHOT,
            "num_candidates": NUM_CANDIDATES,
            "num_selected": TOP_K,
            "selection_ratio": TOP_FRACTION,
            "projection_dim": PROJECTION_DIM,
            "projection_type": "rademacher",
            "projector_seed": PROJECTOR_SEED,
            "projector_model_id": PROJECTOR_MODEL_ID,
            "candidate_features_normalized": True,
            "target_features_normalized": True,
            "epoch_avg_lr": EPOCH_AVG_LR,
            "mmlu_subjects": MMLU_SUBJECTS,
        },
        f,
        indent=2,
    )


print()
print("Saved:")
print(" ", selected_path)
print(" ", scores_path)
print(" ", metadata_path)
print(
    " ",
    NORMALIZED_DATASTORE_ROOT,
)


FINAL LESS SELECTION
Candidates       : 9000
Selection ratio  : 5.0%
Selected         : 450
Score min/max    : 0.000003 / 0.000015
Selected min/max : 0.000013 / 0.000015

Saved:
  /kaggle/working/phase3_selection/less_selected_top450.jsonl
  /kaggle/working/phase3_selection/less_scores_all_9000.npz
  /kaggle/working/phase3_selection/less_selection_metadata.json
  /kaggle/working/phase3_selection/normalized_datastore
